In [ ]:
# Installed to scanpy environment in terminal: pip install decoupler

In [ ]:
import scanpy as sc
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import decoupler as dc
import os

sc.set_figure_params(figsize=(4, 4))
date = "DATE"

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
# Collect annotation info for epithelial cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Epithelial_annotation.h5ad'))
annot = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot['Annotation_Tier1_check']='Epithelial'

In [ ]:
# Collect annotation info for T cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_T_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='T'
annot = pd.concat([annot, annot_tmp])

In [ ]:
# Collect annotation info for B cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_B_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='B'
annot = pd.concat([annot, annot_tmp])

In [ ]:
# Collect annotation info for Stromal cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Stromal_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='Stromal'
annot = pd.concat([annot, annot_tmp])

In [ ]:
# Collect annotation info for Myeloid cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Myeloid_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='Myeloid'
annot = pd.concat([annot, annot_tmp])

In [ ]:
# Collect annotation info for Endothelial cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Endothelial_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='Endothelial'
annot = pd.concat([annot, annot_tmp])

In [ ]:
# Collect annotation info for Glial_Neuronal cells 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Glial_Neuronal_annotation.h5ad'))
annot_tmp = pd.DataFrame(adata.obs['Annotation_Tier2'])
annot_tmp['Annotation_Tier1_check']='Glial_Neuronal'
annot = pd.concat([annot, annot_tmp])

# Add annotation information to raw data object

In [ ]:
# Load in all raw data
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withAnnotation.h5ad'))

In [ ]:
annot = annot.loc[adata.obs.index]
annot

In [ ]:
# check alignment 
print(sum(annot.index==adata.obs.index))

# add annot information 
adata.obs = pd.concat([adata.obs, annot], axis=1)

# Save raw object with annotation information 
adata.write_h5ad(os.path.join(base_path, 'data/all_samples_raw_withTier2Annotation.h5ad'))

# Add annotation information to full processed object

In [ ]:
# Load in processed adata and merge Annotation columns into .obs
adata_proc = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_annotated.h5ad'))
print(sum(annot.index==adata_proc.obs.index))
adata_proc.obs = pd.concat([adata_proc.obs, annot], axis=1)
adata_proc.write_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))

# create sample counts files 

In [ ]:
# Jump in point 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withTier2Annotation.h5ad'))

In [ ]:
# Create counts files for compositional analysis for Tier2 annotation 
grouped_counts = adata.obs.groupby('FRID')['Annotation_Tier2'].value_counts()
grouped_counts_matrix = grouped_counts.unstack(fill_value=0)
grouped_counts_matrix.T.to_csv(os.path.join(base_path, f'results/Annotation_Tier2_counts_{date}.csv'))

In [ ]:
# Create counts files for compositional analysis for Tier1 annotation 
grouped_counts = adata.obs.groupby('FRID')['Annotation_Tier1'].value_counts()
grouped_counts_matrix = grouped_counts.unstack(fill_value=0)
grouped_counts_matrix.T.to_csv(os.path.join(base_path, f'results/Annotation_Tier1_counts_{date}.csv'))

In [ ]:
adata.obs['Annotation_Tier1Tier2'] = adata.obs['Annotation_Tier1'].astype(str) + '_' + adata.obs['Annotation_Tier2'].astype(str)
grouped_counts = adata.obs.groupby('FRID')['Annotation_Tier1Tier2'].value_counts()
grouped_counts_matrix = grouped_counts.unstack(fill_value=0)
grouped_counts_matrix.T.to_csv(os.path.join(base_path, f'results/Annotation_Tier1Tier2_counts_{date}.csv'))

# Pseudobulk the data by Tier2 annotation 

In [ ]:
# Load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withTier2Annotation.h5ad'))

In [ ]:
# check annotation levels 
np.unique(adata.obs['Annotation_Tier2'])

In [ ]:
# Pseudobulk the Tier 2 data 
pdata = dc.get_pseudobulk(
    adata,
    sample_col='FRID',
    groups_col='Annotation_Tier2',
    #layer='counts',
    mode='sum',
    min_cells=10,
    min_counts=1000
)
pdata.obs['pred_dbl']='ignore_me'

In [ ]:
# save Tier2 pseudobulked object 
pdata.write_h5ad(os.path.join(base_path, f'data/pseudobulk_Tier2annotation_{date}.h5ad'))

In [ ]:
# Store raw counts in layers
pdata.layers['counts'] = pdata.X.copy()

# Normalize, scale and compute pca for any analysis with pseudobulk object 
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)

# Return raw counts to X
dc.swap_layer(pdata, 'counts', X_layer_key=None, inplace=True)

In [ ]:
# save files for DESeq2
clusters = np.unique(adata.obs['Annotation_Tier2'])
# Create the directory if it doesn't exist
os.makedirs(os.path.join(base_path, f"results/{date}_YOCRC_pseudobulk"), exist_ok=True)

for i in clusters: 
   
    subset = pdata[pdata.obs['Annotation_Tier2'] == i].copy()
    i_rename = i.replace("/", "").replace(" ", "_")
    
    # Test genes that are expressed in at least 10% of cells in a condition (or 10% of samples have over 10 count?)
    young = subset[subset.obs['Cohort']=='UnderFifty']
    genes_young = dc.filter_by_expr(young, min_prop=0.1, min_count=10)
    old = subset[subset.obs['Cohort']=='FiftyPlus']
    genes_old = dc.filter_by_expr(old, min_prop=0.1, min_count=10)

    genes = []
    genes.extend(genes_young)
    genes.extend(genes_old)
    genes = np.unique(genes)
    
    subset = subset[:, genes].copy()

    # Save pseudbulked results to .csv files 
    np.savetxt(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_{i_rename}.csv'), subset.X, delimiter=",")
    subset.obs.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_{i_rename}_obs.csv'))
    subset.var.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_{i_rename}_var.csv'))

# Pseudobulk the data by Tier1 annotation 

In [ ]:
# Load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withTier2Annotation.h5ad'))

In [ ]:
# Check annotation 
np.unique(adata.obs['Annotation_Tier1'])

In [ ]:
# Pseudobulk the data 
pdata = dc.get_pseudobulk(
    adata,
    sample_col='FRID',
    groups_col='Annotation_Tier1',
    mode='sum',
    min_cells=10,
    min_counts=1000
)
pdata.obs['pred_dbl']='ignore_me'

In [ ]:
# save pseudobulk 
pdata.write_h5ad(os.path.join(base_path, f'data/pseudobulk_Tier1annotation_{date}.h5ad'))

In [ ]:
# Store raw counts in layers
pdata.layers['counts'] = pdata.X.copy()

# Normalize, scale and compute pca for any pseudobulk analysis 
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)

# Return raw counts to X
dc.swap_layer(pdata, 'counts', X_layer_key=None, inplace=True)

In [ ]:
dc.plot_associations(
    pdata,
    uns_key='pca_anova',  # Summary statistics from the anova tests
    obsm_key='X_pca',  # where the PCs are stored
    stat_col='p_adj',  # Which summary statistic to plot
    obs_annotation_cols = ['Cohort'], # which sample annotations to plot
    titles=['Principle component scores', 'Adjusted p-values from ANOVA'],
    figsize=(8, 7),
    n_factors=10,
)

In [ ]:
clusters = np.unique(adata.obs['Annotation_Tier1'])
os.makedirs(os.path.join(base_path, f"results/{date}_YOCRC_pseudobulk"), exist_ok=True)

for i in clusters: 
   
    subset = pdata[pdata.obs['Annotation_Tier1'] == i].copy()
    i_rename = i.replace("/", "").replace(" ", "_")
    
    # Test genes that are expressed in at least 10% of cells in a condition (or 10% of samples have over 10 count?)
    young = subset[subset.obs['Cohort']=='UnderFifty']
    genes_young = dc.filter_by_expr(young, min_prop=0.1, min_count=10)
    old = subset[subset.obs['Cohort']=='FiftyPlus']
    genes_old = dc.filter_by_expr(old, min_prop=0.1, min_count=10)

    genes = []
    genes.extend(genes_young)
    genes.extend(genes_old)
    genes = np.unique(genes)
    
    subset = subset[:, genes].copy()
    #subset.write("/home/jupyter/Young_Onset_CRC/results/FINAL/epi_pseudobulk_May30/YOCRC_integrated_epithelial_lgr5_nn_PSEUDOBULKED.h5ad")

    # Save pseudbulked results to .csv files 
    np.savetxt(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_Tier1_{i_rename}.csv'), subset.X, delimiter=",")
    subset.obs.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_Tier1_{i_rename}_obs.csv'))
    subset.var.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_Tier1_{i_rename}_var.csv'))

# Pseudobulk all of the data across all cell types (to create bulkRNA-Seq type data)

In [ ]:
# load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withTier2Annotation.h5ad'))

In [ ]:
# Add column to metadata to use as pseudobulk group 
adata.obs['Project'] = "YOCRC"

In [ ]:
# Pseudobulk the data 
pdata = dc.get_pseudobulk(
    adata,
    sample_col='FRID',
    groups_col='Project',
    mode='sum',
    min_cells=10,
    min_counts=1000
)
pdata.obs['pred_dbl']='ignore_me'

In [ ]:
# Save pseudbulked results to .csv files 
np.savetxt(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_allCellTypes.csv'), pdata.X, delimiter=",")
pdata.obs.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_allCellTypes_obs.csv'))
pdata.var.to_csv(os.path.join(base_path, f'results/{date}_YOCRC_pseudobulk/pseudobulk_{date}_allCellTypes_var.csv'))